## Project Description

For the Words to Embeddings group work assignment, we decided to create word embeddings on a small-, medium-, and large-sized dataset that builds off the same corpus of text and then test to see how relationships between words differ between the sets of vectors. The corpus of text we used is The Complete Works of Shakespeare pulled from Project Gutenberg. The small dataset consists of the first 25% of the plain text file, the medium dataset consists of the first 50% of the file, and the large dataset is the entire file.

After all of the embeddings were created and trained with each sized dataset, we tested to see how certain relationships between words differed. The testing consisted of using the vector offset method to get the word embedding for "queen" from the expression "king - man + woman" and seeing how off each set of embeddings was from the true vector for "queen". 

## Results

In [2]:
from gensim.utils import simple_preprocess
from pathlib import Path
import nltk
from nltk.tokenize import sent_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/dsu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
def create_sentence_tokens(text):
    sentences = sent_tokenize(text) 
    sentence_tokens = [simple_preprocess(sent) for sent in sentences]
    return sentence_tokens

def create_model(title, sentences):
    model = Word2Vec(
        sentences=sentences,
        vector_size=50,   # embedding dimension
        window=5,
        min_count=2,       # ignore rare words (adjust to your corpus size)
        sg=1,              # 1 = skip-gram, 0 = CBOW
        negative=10,
        sample=1e-4,
        epochs=500
    )

    model.save(f'{title}.model')

In [4]:
model = Word2Vec.load("Shakespeare.model")

import numpy as np

vec = model.wv['king'] - model.wv['man'] + model.wv['woman']

# find closest word
similar = model.wv.similar_by_vector(vec, topn=5)
print(similar)


[('king', 0.8143742680549622), ('queen', 0.780026376247406), ('son', 0.7672947645187378), ('edward', 0.7084923982620239), ('henry', 0.7077077031135559)]


In [5]:
text = Path("Shakespeare.txt").read_text(encoding='utf-8', errors='ignore')

sentence_tokens = create_sentence_tokens(text)

create_model("lg", sentence_tokens)

In [6]:
file_path = "Shakespeare.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# take first 25%
cutoff = int(len(lines) * 0.25)
sm_text = "".join(lines[:cutoff])

sentence_tokens = create_sentence_tokens(sm_text)

create_model("sm", sentence_tokens)

In [7]:
file_path = "Shakespeare.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# take first 50%
cutoff = int(len(lines) * 0.50)
md_text = "".join(lines[:cutoff])

sentence_tokens = create_sentence_tokens(md_text)

create_model("md", sentence_tokens)

In [8]:
vec = model.wv['king'] - model.wv['man'] + model.wv['woman']

model = Word2Vec.load("sm.model")
queen = model.wv['queen']
similarity = cosine_similarity([vec], [queen])[0][0]
print(f"Small text model: {similarity}")

model = Word2Vec.load("md.model")
queen = model.wv['queen']
similarity = cosine_similarity([vec], [queen])[0][0]
print(f"Medium text model: {similarity}")

model = Word2Vec.load("lg.model")
queen = model.wv['queen']
similarity = cosine_similarity([vec], [queen])[0][0]
print(f"Large text model: {similarity}")

Small text model: 0.06822717189788818
Medium text model: 0.2876628041267395
Large text model: 0.5940263271331787
